# override_field_config

用于装饰 `Mycords` 类型及其子类，两种使用方式
- 直接装饰
  ```python
  @override_field_config
  class MyRecords(Records):
    pass
  ```
- 带参数装饰
  ```python
    custom_config = {
        'dtype': order_dtype,
        'settings': {
            'side': {'title': '买卖方向', 'mapping': OrderSide},
            'status': {'title': '订单状态', 'mapping': OrderStatus}
        }
    }
    
    @override_field_config(custom_config)
    class OrderRecords(Records):
      pass
  ```

作用是设置装饰的类 `cls` 的配置项 `_field_config`：
- 是否带参数决定合并配置项过程中，优先级最高的是 `cls._field_config` 还是传入的参数
- `merge_configs` 决定是否合并 `cls` 父类中那些同为 `Records` 子类的 `_field_config`


```python
def override_field_config(*args, merge_configs: bool = True) -> tp.Union[WrapperFuncT, tp.Type[tp.T]]:

    def wrapper(cls: tp.Type[tp.T], config: tp.DictLike = None) -> tp.Type[tp.T]:
        checks.assert_subclass_of(cls, "Records")

        if config is None:
            config = cls.field_config
        if not isinstance(config, Config):
            config = Config(config, readonly=True, as_attrs=False)
        if merge_configs:
            configs = []
            # 遍历类的方法解析顺序（MRO）
            for base_cls in cls.mro()[::-1]: # 反转顺序    
                if base_cls is not cls: # 跳过 cls 本身
                    if checks.is_subclass_of(base_cls, "Records"):
                        # 收集同样继承了 Records 的记录基类的 field_config
                        configs.append(base_cls.field_config)
            # 添加 cls 的配置到最后（优先级最高）
            configs.append(config)
            # 合并所有 field_config
            config = merge_dicts(*configs, to_dict=False)

        setattr(cls, "_field_config", config)
        return cls

    if len(args) == 0:
        return wrapper
    elif len(args) == 1:
        if isinstance(args[0], type):
            return wrapper(args[0])
        return partial(wrapper, config=args[0])
    elif len(args) == 2:
        return wrapper(args[0], config=args[1])
    raise ValueError("Either class, config, class and config, or keyword arguments must be passed")
```

# attach_fields
用于装饰 `Mycords` 类型及其子类。

```python
def attach_fields(*args, on_conflict: str = 'raise') -> tp.Union[WrapperFuncT, tp.Type[tp.T]]:

    def wrapper(cls: tp.Type[tp.T], config: tp.DictLike = None) -> tp.Type[tp.T]:
        checks.assert_subclass_of(cls, "Records")

        dtype = cls.field_config.get('dtype', None)
        checks.assert_not_none(dtype.fields)

        if config is None:
            config = {}

        def _prepare_attr_name(attr_name: str) -> str:
            checks.assert_instance_of(attr_name, str)
            attr_name = attr_name.replace('NaN', 'Nan')
            startswith_ = attr_name.startswith('_')
            attr_name = re.sub(r"([A-Z])", r"_\1", attr_name)
            if not startswith_ and attr_name.startswith('_'):
                attr_name = attr_name[1:]
            attr_name = attr_name.lower()
            if keyword.iskeyword(attr_name):
                attr_name += '_'
            return attr_name

        def _check_attr_name(attr_name, _on_conflict: str = on_conflict) -> None:
            if attr_name not in cls.field_config.get('settings', {}):
                # Consider only attributes that are not listed in the field config
                if hasattr(cls, attr_name):
                    if _on_conflict.lower() == 'raise':
                        raise ValueError(f"An attribute with the name '{attr_name}' already exists in {cls}")
                    if _on_conflict.lower() == 'ignore':
                        return
                    if _on_conflict.lower() == 'override':
                        return
                    raise ValueError(f"Value '{_on_conflict}' is invalid for on_conflict")
                if keyword.iskeyword(attr_name):
                    raise ValueError(f"Name '{attr_name}' is a keyword and cannot be used as an attribute name")

        if dtype is not None:
            for field_name in dtype.names:
                settings = config.get(field_name, {})
                attach = settings.get('attach', True)
                if not isinstance(attach, bool):
                    target_name = attach
                    attach = True
                else:
                    target_name = field_name
                defaults = settings.get('defaults', None)
                if defaults is None:
                    defaults = {}
                attach_filters = settings.get('attach_filters', False)
                filter_defaults = settings.get('filter_defaults', None)
                if filter_defaults is None:
                    filter_defaults = {}
                _on_conflict = settings.get('on_conflict', on_conflict)

                if attach:
                    target_name = _prepare_attr_name(target_name)
                    _check_attr_name(target_name, _on_conflict)

                    def new_prop(self,
                                 _field_name: str = field_name,
                                 _defaults: tp.KwargsLike = defaults) -> MappedArray:
                        return self.get_map_field(_field_name, **_defaults)

                    new_prop.__doc__ = f"Mapped array of the field `{field_name}`."
                    new_prop.__name__ = target_name
                    setattr(cls, target_name, cached_property(new_prop))

                if attach_filters:
                    if isinstance(attach_filters, bool):
                        if not attach_filters:
                            continue
                        mapping = cls.field_config \
                            .get('settings', {}) \
                            .get(field_name, {}) \
                            .get('mapping', None)
                    else:
                        mapping = attach_filters
                    if mapping is None:
                        raise ValueError(f"Field '{field_name}': Mapping is required to attach filters")
                    mapping = to_mapping(mapping)

                    for filter_value, target_filter_name in mapping.items():
                        if target_filter_name is None:
                            continue
                        target_filter_name = _prepare_attr_name(target_filter_name)
                        _check_attr_name(target_filter_name, _on_conflict)
                        if target_filter_name in filter_defaults:
                            __filter_defaults = filter_defaults[target_filter_name]
                        else:
                            __filter_defaults = filter_defaults

                        def new_filter_prop(self,
                                            _field_name: str = field_name,
                                            _filter_value: tp.Any = filter_value,
                                            _filter_defaults: tp.KwargsLike = __filter_defaults) -> MappedArray:
                            filter_mask = self.get_field_arr(_field_name) == _filter_value
                            return self.apply_mask(filter_mask, **_filter_defaults)

                        new_filter_prop.__doc__ = f"Records filtered by `{field_name} == {filter_value}`."
                        new_filter_prop.__name__ = target_filter_name
                        setattr(cls, target_filter_name, cached_property(new_filter_prop))

        return cls

    if len(args) == 0:
        return wrapper
    elif len(args) == 1:
        if isinstance(args[0], type):
            return wrapper(args[0])
        return partial(wrapper, config=args[0])
    elif len(args) == 2:
        return wrapper(args[0], config=args[1])
    raise ValueError("Either class, config, class and config, or keyword arguments must be passed")
```

# 例子

## 订单数据结构

In [ ]:
import numpy as np
import pandas as pd
import vectorbt as vbt
from vectorbt.records.decorators import attach_fields, override_field_config
from vectorbt.utils.config import Config

# 定义订单方向枚举
class OrderSide:
    BUY = 0
    SELL = 1

# 定义订单状态枚举
class OrderStatus:
    PENDING = 0 # 待处理
    FILLED = 1 # 已成交
    CANCELLED = 2 # 已取消
    PARTIAL = 3 # 部分成交

# 定义订单类型枚举
class OrderType:
    MARKET = 0 # 市价单
    LIMIT = 1 # 限价单
    STOP = 2 # 止损单

# 定义订单数据结构
order_dtype = np.dtype([
    ('id', np.int64),           # 订单ID
    ('col', np.int64),          # 列索引（股票）
    ('idx', np.int64),          # 时间索引
    ('price', np.float64),      # 订单价格
    ('quantity', np.float64),   # 订单数量
    ('side', np.int8),          # 买卖方向
    ('status', np.int8),        # 订单状态
    ('order_type', np.int8),    # 订单类型
    ('fees', np.float64),       # 手续费
    ('filled_qty', np.float64)  # 已成交数量
])

## 字段配置

In [ ]:
# 字段配置：定义字段的映射关系
order_field_config = Config({
    'dtype': order_dtype,
    'settings': {
        'id': {'name': 'id', 'title': '订单ID'},
        'col': {'name': 'col', 'title': '股票', 'mapping': 'columns'},
        'idx': {'name': 'idx', 'title': '时间', 'mapping': 'index'},
        'price': {'name': 'price', 'title': '价格'},
        'quantity': {'name': 'quantity', 'title': '数量'},
        'side': {'name': 'side', 'title': '方向', 'mapping': {
            OrderSide.BUY: 'Buy',
            OrderSide.SELL: 'Sell'
        }},
        'status': {'name': 'status', 'title': '状态', 'mapping': {
            OrderStatus.PENDING: 'Pending',
            OrderStatus.FILLED: 'Filled',
            OrderStatus.CANCELLED: 'Cancelled',
            OrderStatus.PARTIAL: 'Partial'
        }},
        'order_type': {'name': 'order_type', 'title': '类型', 'mapping': {
            OrderType.MARKET: 'Market',
            OrderType.LIMIT: 'Limit',
            OrderType.STOP: 'Stop'
        }},
        'fees': {'name': 'fees', 'title': '手续费'},
        'filled_qty': {'name': 'filled_qty', 'title': '已成交量'}
    }
})

## attach_fields 配置：要自动生成的属性和过滤器

In [ ]:
# attach_fields配置：定义要自动生成的属性和过滤器
order_attach_config = Config({
    # 价格字段：生成普通属性，带默认参数
    'price': {
        'attach': True,                    # 生成price属性
        'defaults': {'normalize': False}   # 默认参数
    },
    
    # 数量字段：生成普通属性
    'quantity': {
        'attach': True
    },
    
    # 手续费字段：生成自定义名称的属性
    'fees': {
        'attach': 'transaction_costs',     # 使用自定义属性名
        'defaults': {'round_digits': 4}    # 默认参数
    },
    
    # 方向字段：生成过滤器
    'side': {
        'attach': True,                    # 生成side属性
        'attach_filters': True,            # 生成过滤器
        'filter_defaults': {}              # 过滤器默认参数
    },
    
    # 状态字段：生成过滤器，带特定默认参数
    'status': {
        'attach': True,                    # 生成status属性
        'attach_filters': True,            # 生成过滤器
        'filter_defaults': {               # 不同过滤器的默认参数
            'filled': {'include_fees': True},
            'pending': {'include_fees': False},
            'cancelled': {'include_fees': False},
            'partial': {'include_fees': True}
        }
    },
    
    # 订单类型字段：生成过滤器
    'order_type': {
        'attach': True,                    # 生成order_type属性
        'attach_filters': True,            # 生成过滤器
        'filter_defaults': {}              # 过滤器默认参数
    },
    
    # 已成交量字段：生成自定义属性
    'filled_qty': {
        'attach': 'executed_quantity',     # 使用自定义属性名
        'defaults': {'min_periods': 1}     # 默认参数
    }
})

## 定义 Records 类并应用装饰器

In [ ]:
# 注意：装饰器的顺序很重要！
# 1. 先使用 @attach_fields 生成属性和过滤器
# 2. 再使用 @override_field_config 设置字段配置
@attach_fields(order_attach_config)
@override_field_config(order_field_config)
class OrderRecords(vbt.Records):
    """
    订单记录类 - 演示attach_fields装饰器的使用
    
    这个类演示了attach_fields装饰器如何自动生成：
    1. 字段属性访问器
    2. 基于字段值的过滤器方法
    3. 自定义命名和参数设置
    """
    
    def __init__(self, wrapper, records_arr, **kwargs):
        super().__init__(wrapper, records_arr, **kwargs)
    
    # 自定义方法：计算总交易价值
    def total_value(self):
        """计算总交易价值"""
        return self.map_array(
            self.get_field_arr('price') * self.get_field_arr('quantity')
        )
    
    # 自定义方法：获取活跃订单
    def active_orders(self):
        """获取活跃订单（待处理和部分成交）"""
        active_mask = np.isin(
            self.get_field_arr('status'), 
            [OrderStatus.PENDING, OrderStatus.PARTIAL]
        )
        return self.apply_mask(active_mask)

## 创建测试数据

In [ ]:
# 创建时间索引
time_index = pd.date_range('2023-01-01', periods=10, freq='D')

# 创建股票列名
stock_columns = ['AAPL', 'GOOGL', 'MSFT', 'TSLA']

# 创建ArrayWrapper
wrapper = vbt.ArrayWrapper(
    index=time_index,
    columns=stock_columns,
    ndim=2,
    freq='D'
)

# 创建订单数据
order_data = np.array([
    # (id, col, idx, price, quantity, side, status, order_type, fees, filled_qty)
    (1, 0, 0, 150.0, 100, OrderSide.BUY, OrderStatus.FILLED, OrderType.MARKET, 1.5, 100),
    (2, 0, 1, 152.0, 50, OrderSide.SELL, OrderStatus.FILLED, OrderType.LIMIT, 0.76, 50),
    (3, 1, 0, 2800.0, 10, OrderSide.BUY, OrderStatus.FILLED, OrderType.MARKET, 28.0, 10),
    (4, 1, 2, 2850.0, 5, OrderSide.SELL, OrderStatus.PENDING, OrderType.LIMIT, 0.0, 0),
    (5, 2, 1, 380.0, 200, OrderSide.BUY, OrderStatus.FILLED, OrderType.MARKET, 7.6, 200),
    (6, 2, 3, 385.0, 100, OrderSide.SELL, OrderStatus.PARTIAL, OrderType.LIMIT, 1.925, 50),
    (7, 3, 2, 220.0, 150, OrderSide.BUY, OrderStatus.CANCELLED, OrderType.STOP, 0.0, 0),
    (8, 3, 4, 225.0, 100, OrderSide.BUY, OrderStatus.FILLED, OrderType.MARKET, 2.25, 100),
    (9, 0, 5, 148.0, 200, OrderSide.BUY, OrderStatus.PENDING, OrderType.LIMIT, 0.0, 0),
    (10, 1, 6, 2900.0, 8, OrderSide.SELL, OrderStatus.FILLED, OrderType.MARKET, 23.2, 8)
], dtype=order_dtype)

# 创建OrderRecords实例
orders = OrderRecords(wrapper, order_data)

## 功能演示

### 基本信息

In [ ]:
# 6.1 基本信息
print(f"订单总数: {len(orders)}")
print(f"时间范围: {orders.wrapper.index[0]} 到 {orders.wrapper.index[-1]}")
print(f"股票数量: {len(orders.wrapper.columns)}")

### 查看实际的列名

In [ ]:
# 6.2 查看实际的列名
print("\n2. 查看实际的列名:")
print("records_readable 的列名:", orders.records_readable.columns.tolist())
print("records 的列名:", orders.records.columns.tolist())

### 自动生成的字段属性

In [ ]:
dtype = type(orders).field_config.get('dtype', None)
for field in dtype.names:
    print(field)

# 6.3 自动生成的字段属性
print("\n3. 自动生成的字段属性:")
print("3.1 价格属性 (orders.price):")
print(orders.price.values)

print("\n3.2 数量属性 (orders.quantity):")
print(orders.quantity.values)

### 自动生成的过滤器方法（使用正确的列名）

In [ ]:
# 6.4 自动生成的过滤器方法（使用正确的列名）
print("\n4. 自动生成的过滤器方法:")

# 6.4.1 方向过滤器
print("4.1 买入订单 (orders.buy):")
buy_orders = orders.buy
print(f"买入订单数量: {len(buy_orders)}")
print(buy_orders.records_readable)
print(buy_orders.records_readable[['订单ID', '股票', '时间', '价格', '数量', '方向']])

print("\n4.2 卖出订单 (orders.sell):")
sell_orders = orders.sell
print(f"卖出订单数量: {len(sell_orders)}")
print(sell_orders.records_readable[['订单ID', '股票', '时间', '价格', '数量', '方向']])

# 6.4.2 状态过滤器
print("\n4.3 已成交订单 (orders.filled):")
filled_orders = orders.filled
print(f"已成交订单数量: {len(filled_orders)}")
print(filled_orders.records_readable[['订单ID', '股票', '状态', '价格', '数量']])

print("\n4.4 待处理订单 (orders.pending):")
pending_orders = orders.pending
print(f"待处理订单数量: {len(pending_orders)}")
print(pending_orders.records_readable[['订单ID', '股票', '状态', '价格', '数量']])

print("\n4.5 部分成交订单 (orders.partial):")
partial_orders = orders.partial
print(f"部分成交订单数量: {len(partial_orders)}")
print(partial_orders.records_readable[['订单ID', '股票', '状态', '价格', '数量', '已成交量']])

# 6.4.3 订单类型过滤器
print("\n4.6 市价订单 (orders.market):")
market_orders = orders.market
print(f"市价订单数量: {len(market_orders)}")
print(market_orders.records_readable[['订单ID', '股票', '类型', '价格', '数量']])

print("\n4.7 限价订单 (orders.limit):")
limit_orders = orders.limit
print(f"限价订单数量: {len(limit_orders)}")
print(limit_orders.records_readable[['订单ID', '股票', '类型', '价格', '数量']])

In [ ]:
# ================================================================
# 第八步：高级功能演示
# ================================================================

print("\n" + "=" * 80)
print("高级功能演示")
print("=" * 80)

# 8.1 缓存机制
print("\n1. 缓存机制:")
print("生成的属性都使用 @cached_property 装饰器，避免重复计算")

import time

start_time = time.time()
prices1 = orders.price
end_time = time.time()
first_call_time = end_time - start_time

start_time = time.time()
prices2 = orders.price
end_time = time.time()
second_call_time = end_time - start_time

print(f"第一次调用 orders.price: {first_call_time:.6f} 秒")
print(f"第二次调用 orders.price: {second_call_time:.6f} 秒")
print(f"缓存提升: {first_call_time / second_call_time:.2f}x")

# 8.2 属性链式调用
print("\n2. 属性链式调用:")
print("可以将过滤器和属性进行链式调用")

# 获取已成交买入订单的平均价格
avg_buy_filled_price = orders.buy.filled.price.mean()
print(f"已成交买入订单平均价格: {avg_buy_filled_price}")

# 获取限价卖出订单的总数量
total_limit_sell_qty = orders.sell.limit.quantity.sum()
print(f"限价卖出订单总数量: {total_limit_sell_qty}")

# 8.3 分组分析
print("\n3. 分组分析:")
print("按行业分组分析（假设前两只股票是科技股）")

# 按行业分组
industry_groups = ['Tech', 'Tech', 'Finance', 'Auto']
industry_order_counts = orders.count().groupby(industry_groups).sum()
print(f"按行业的订单数量: {industry_order_counts}")

industry_avg_prices = orders.price.mean().groupby(industry_groups).mean()
print(f"按行业的平均价格: {industry_avg_prices}")

print("\n" + "=" * 80)
print("演示完成！")
print("=" * 80)

# 关于例子的说明

先定义数据结构：
```python
order_dtype = np.dtype([
    ('id', np.int64),           # 订单ID
    ('col', np.int64),          # 列索引（股票）
    ('idx', np.int64),          # 时间索引
    ('price', np.float64),      # 订单价格
    ('quantity', np.float64),   # 订单数量
    ('side', np.int8),          # 买卖方向
    ('status', np.int8),        # 订单状态
    ('order_type', np.int8),    # 订单类型
    ('fees', np.float64),       # 手续费
    ('filled_qty', np.float64)  # 已成交数量
])
```

然后定义数据结构映射：包括每个字段的名称 `name`，显示标题 `title`，以及映射或取值范围 `mapping`。
```python
# 字段配置：定义字段的映射关系
order_field_config = Config({
    'dtype': order_dtype,
    'settings': {
        'id': {'name': 'id', 'title': '订单ID'},
        'col': {'name': 'col', 'title': '股票', 'mapping': 'columns'},
        'idx': {'name': 'idx', 'title': '时间', 'mapping': 'index'},
        'price': {'name': 'price', 'title': '价格'},
        'quantity': {'name': 'quantity', 'title': '数量'},
        'side': {'name': 'side', 'title': '方向', 'mapping': {
            OrderSide.BUY: 'Buy',
            OrderSide.SELL: 'Sell'
        }},
        'status': {'name': 'status', 'title': '状态', 'mapping': {
            OrderStatus.PENDING: 'Pending',
            OrderStatus.FILLED: 'Filled',
            OrderStatus.CANCELLED: 'Cancelled',
            OrderStatus.PARTIAL: 'Partial'
        }},
        'order_type': {'name': 'order_type', 'title': '类型', 'mapping': {
            OrderType.MARKET: 'Market',
            OrderType.LIMIT: 'Limit',
            OrderType.STOP: 'Stop'
        }},
        'fees': {'name': 'fees', 'title': '手续费'},
        'filled_qty': {'name': 'filled_qty', 'title': '已成交量'}
    }
})
```

然后定义：每个字段是否生成属性/可过滤的属性（根据取值）。
```python
order_attach_config = Config({
    # 价格字段：生成普通属性，带默认参数
    'price': {
        'attach': True,                    # 生成price属性
        'defaults': {'normalize': False}   # 默认参数
    },
    
    # 数量字段：生成普通属性
    'quantity': {
        'attach': True
    },
    
    # 手续费字段：生成自定义名称的属性
    'fees': {
        'attach': 'transaction_costs',     # 使用自定义属性名
        'defaults': {'round_digits': 4}    # 默认参数
    },
    
    # 方向字段：生成过滤器
    'side': {
        'attach': True,                    # 生成side属性
        'attach_filters': True,            # 生成过滤器
        'filter_defaults': {}              # 过滤器默认参数
    },
    
    # 状态字段：生成过滤器，带特定默认参数
    'status': {
        'attach': True,                    # 生成status属性
        'attach_filters': True,            # 生成过滤器
        'filter_defaults': {               # 不同过滤器的默认参数
            'filled': {'include_fees': True},
            'pending': {'include_fees': False},
            'cancelled': {'include_fees': False},
            'partial': {'include_fees': True}
        }
    },
    
    # 订单类型字段：生成过滤器
    'order_type': {
        'attach': True,                    # 生成order_type属性
        'attach_filters': True,            # 生成过滤器
        'filter_defaults': {}              # 过滤器默认参数
    },
    
    # 已成交量字段：生成自定义属性
    'filled_qty': {
        'attach': 'executed_quantity',     # 使用自定义属性名
        'defaults': {'min_periods': 1}     # 默认参数
    }
})
```

最后定义类：
```python
# 注意：装饰器的顺序很重要！
@attach_fields(order_attach_config)
@override_field_config(order_field_config)
class OrderRecords(vbt.Records): ...
```
